<a href="https://colab.research.google.com/github/720-hz/flyrank-ml-internship/blob/main/work/notebooks/Week%203/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/720-hz/flyrank-ml-internship/blob/main/work/notebooks/Week%203/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Unit of analysis + time window

**One row (in the raw warehouse fact table) = one content item, for one client, on one report date** — the grain of `fact_content_daily_performance` is `report_date x client_hash_id x content_hash_id`. That is finer than the decision I actually make: I score content items for review on a weekly cadence, so my *decision-grain* row (built below in section 3) is one content item, as of one chosen decision date, summarizing a trailing window of those daily rows.

**Table(s):** `fact_content_daily_performance` (the daily panel — trailing features and the forward-window label live here) joined to `dim_content` (content-level static attributes, e.g. `content_created_date`) and `dim_clients` (per-client history bounds: `gsc_data_start`, `ga4_data_start` — required before trusting any date window, per the panel warning).

**Time window:** I iterate on **`month=2026-03`**, a mid-panel month, exactly as instructed — the `_sample` table is the sealed final month (June 2026) and is for query-mechanics testing only, never for label logic, since it *is* the natural outcome window of any past-to-future label. The full daily panel spans 2025-01-27 to 2026-06-30 (~17 months, unbalanced across clients).

**What I'd predict or rank (label or proxy):** same lane as ML-03 — a priority queue for a content strategist's weekly review capacity (Precision@K, K=20/50). The label is a **future-observed decline**: whether a content item's search demand (impressions) in a short window *after* a decision date is lower than its demand in the window immediately *before* it. This is a real improvement over ML-03's placeholder proxy (`trend_direction`, a static bucket in the 90-day-trailing starter CSV) — the daily panel lets me build an actual forward-looking outcome instead of a backward-looking stand-in, which is exactly the gap I flagged on purpose back in ML-03.

**One thing I deliberately exclude:** any FlyRank product-computed decision field (`health_score`, `priority_score`, `action_type`, refresh flags). They aren't shipped in this release at all, and the reason matters as much as the fact: feeding a system's own decision into a model that's supposed to *discover* priority just teaches the model to copy that decision — a circular result, not a finding.


In [3]:
%pip -q install duckdb
import duckdb, os
from getpass import getpass

con = duckdb.connect()

# HF token: getpass in Colab (or the notebook's own Secrets panel) -- NEVER pasted into a cell,
# this repo is public. In Colab: use the key-icon Secrets panel and name the secret HF_TOKEN.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("HF_TOKEN (read token, gated-repo access requested): ")

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

MONTH = "2026-03"  # mid-panel month -- never the _sample table for label logic
DAILY = f"read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet')"

# Session sanity check -- COUNT(*) and MIN/MAX(date) touch Parquet metadata, not data: near-free.
sanity = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {DAILY}
""").df()
sanity


HF_TOKEN (read token, gated-repo access requested): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

First, the real schema (not memory) — `DESCRIBE` the table before bucketing a single field:


In [8]:
schema = con.sql(f"DESCRIBE SELECT * FROM {DAILY} LIMIT 0").df()
schema


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [9]:
print(schema.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

**Context (join / grouping / splitting only — never a feature):** `content_hash_id`, `client_hash_id`, `report_date` (defines the window, not a predictor itself), plus `keyword_hash_id` / `url_hash_id` from `dim_content` for case analysis. Pseudonyms carry no signal of their own.

**Feature (knowable strictly before the decision date):** trailing-window aggregates of `gsc_avg_position`, GSC impressions/clicks (and derived CTR), GA4 sessions/engagement — every one gated to rows dated *before* the decision date, and every GA4-based one additionally gated on `ga4_data_available IS TRUE`. Content age at the decision date (from `dim_content.content_created_date`) is also a legal feature — static, no future information.

**Label / proxy (never a feature):** the forward-window demand-change flag described in section 1 — computed from `report_date`s *after* the decision date. Its own inputs (the post-decision impressions) can never also appear as a feature; that's exactly the trap I stage on purpose in section 3.

**Excluded, with why:**
- FlyRank product decision fields (`health_score`, `priority_score`, `action_type`) — not shipped in this release; would be circular if they were (they encode a decision, not an observation).
- Raw client names, domains, URLs, queries, titles — not shipped; public-safe-output rule either way.
- Rows before a client's `ga4_data_start` — GA4 columns are zero-filled with `ga4_data_available = FALSE` there; treating those zeros as "no engagement" instead of "not tracked yet" would be a real, easy-to-make error, so they're filtered out of any GA4-feature computation, not silently kept.


## 3. Verify it with queries (grain, counts, availability) — then five features, then the trap

Three small, real queries on `month=2026-03`. Each proves one claim above; a claim without a query next to it is a guess.


### 3a. Grain — one row really is (content x client x day)

In [4]:
grain_probe = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {DAILY}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Duplicate (date, client, content) combinations found: {len(grain_probe)} (0 rows back confirms the grain holds)")
grain_probe


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (date, client, content) combinations found: 0 (0 rows back confirms the grain holds)


,report_date,client_hash_id,content_hash_id,c


### 3b. My slice's row count and date span

In [5]:
slice_stats = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {DAILY}
""").df()
slice_stats


,n_rows,n_content_items,n_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


### 3c. Availability — filter with `IS TRUE`, show how many rows survive

In [6]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_ga4_available
    FROM {DAILY}
""").df()
availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_ga4_available
0,9841378,413966.0,4.21


*Result: total_rows = 9,841,378; ga4_available_rows = 413,966; pct_ga4_available = 4.21%. So the GA4 engagement columns are only meaningfully populated for ~4% of March 2026 rows -- this is the real availability number the "one thing I deliberately exclude" line in section 1 depends on, and it is why every GA4-derived feature below must be gated on `ga4_data_available IS TRUE` rather than treated as zero-filled.*

### 3d. Five features, max — one small feature frame for this lane

Every feature: a trailing aggregate computed strictly from `report_date < decision_date`, at the decision grain (one row per content item as of the chosen decision date). Each gets its own "knowable at the decision moment because..." line.

1. **`gsc_position_trail28`** — mean `gsc_avg_position` over the 28 days before the decision date. *Knowable because* it's built only from `report_date`s strictly before the decision date — nothing from after it enters the average.
2. **`gsc_impressions_trail28`** — summed GSC impressions over the same trailing 28 days. *Knowable because* same trailing-window rule; it's an observed search-demand signal, not a product decision.
3. **`gsc_ctr_trail28`** — trailing clicks / trailing impressions over the same window (0 when impressions are 0). *Knowable because* it's a ratio of two already-legal trailing features, computed at the same cutoff.
4. **`ga4_sessions_trail28`** — trailing GA4 sessions, but only computed where `ga4_data_available IS TRUE` for that row (else left null, not zero). *Knowable because* it's trailing AND explicitly gated on the availability flag from section 2 — a silent zero here would misrepresent "not tracked" as "no engagement."
5. **`content_age_days`** — decision date minus `dim_content.content_created_date`. *Knowable because* it depends only on when the content was published, a fact fixed long before the decision date; joined in from `dim_content`, not the daily table.


In [10]:
DECISION_DATE = "2026-03-22"  # a Sunday inside the mid-panel month, leaves a 28-day trailing window
LOOKBACK_START = "2026-02-22"  # DECISION_DATE - 28 days

CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

feature_frame = con.sql(f"""
    WITH trail AS (
        SELECT
            content_hash_id,
            client_hash_id,
            AVG(gsc_avg_position) AS gsc_position_trail28,
            SUM(gsc_impressions) AS gsc_impressions_trail28,
            SUM(gsc_clicks) AS gsc_clicks_trail28,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE NULL END) AS ga4_sessions_trail28,
            BOOL_OR(ga4_data_available IS TRUE) AS any_ga4_available
        FROM {DAILY}
        WHERE report_date >= DATE '{LOOKBACK_START}' AND report_date < DATE '{DECISION_DATE}'
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        t.content_hash_id,
        t.client_hash_id,
        t.gsc_position_trail28,
        t.gsc_impressions_trail28,
        CASE WHEN t.gsc_impressions_trail28 > 0
             THEN ROUND(100.0 * t.gsc_clicks_trail28 / t.gsc_impressions_trail28, 3)
             ELSE NULL END AS gsc_ctr_trail28,
        CASE WHEN t.any_ga4_available THEN t.ga4_sessions_trail28 ELSE NULL END AS ga4_sessions_trail28,
        DATE '{DECISION_DATE}' - c.content_created_date AS content_age_days
    FROM trail t
    JOIN {CONTENT} c USING (content_hash_id)
""").df()

print(f"Feature frame: {feature_frame.shape[0]} content items x {feature_frame.shape[1]} columns, as of decision date {DECISION_DATE}")
feature_frame.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 324297 content items x 7 columns, as of decision date 2026-03-22


,content_hash_id,client_hash_id,gsc_position_trail28,gsc_impressions_trail28,gsc_ctr_trail28,ga4_sessions_trail28,content_age_days
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,5.793215,143.0,0.000,NaN,38
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,4.900641,26.0,0.000,NaN,38
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,3.568254,63.0,0.000,NaN,38
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,4.595366,280.0,0.000,NaN,38
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,4.553920,309.0,0.647,NaN,38


### 3e. The trap — add one label-derived column on purpose, then remove it

Build the forward-observed label first (strictly *after* `DECISION_DATE`), get an honest quick score with the five features above, then deliberately add a leak, watch the score jump toward perfect, and delete it.


In [11]:
LOOKAHEAD_END = "2026-04-19"  # DECISION_DATE + 28 days

label_frame = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_lookahead28
    FROM {DAILY}
    WHERE report_date >= DATE '{DECISION_DATE}' AND report_date < DATE '{LOOKAHEAD_END}'
    GROUP BY content_hash_id, client_hash_id
""").df()

panel = feature_frame.merge(label_frame, on=["content_hash_id", "client_hash_id"], how="inner")
panel["is_declining"] = (panel["gsc_impressions_lookahead28"] < panel["gsc_impressions_trail28"]).astype(int)
print(f"Panel with a real forward-observed label: {panel.shape[0]} rows, base rate (share declining) = {panel['is_declining'].mean()*100:.1f}%")
panel.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Panel with a real forward-observed label: 324296 rows, base rate (share declining) = 41.2%


,content_hash_id,client_hash_id,gsc_position_trail28,gsc_impressions_trail28,gsc_ctr_trail28,ga4_sessions_trail28,content_age_days,gsc_impressions_lookahead28,is_declining
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,5.793215,143.0,0.000,NaN,38,38.0,1
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,4.900641,26.0,0.000,NaN,38,8.0,1
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,3.568254,63.0,0.000,NaN,38,14.0,1
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,4.595366,280.0,0.000,NaN,38,49.0,1
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,4.553920,309.0,0.647,NaN,38,293.0,1


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

HONEST_FEATURES = ["gsc_position_trail28", "gsc_impressions_trail28", "gsc_ctr_trail28",
                    "ga4_sessions_trail28", "content_age_days"]

def quick_score(df, feature_cols, label_col="is_declining"):
    use = df[feature_cols + [label_col]].dropna()
    X_tr, X_te, y_tr, y_te = train_test_split(
        use[feature_cols], use[label_col], test_size=0.3, random_state=0, stratify=use[label_col]
    )
    clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
    return auc, len(use)

honest_auc, n_used = quick_score(panel, HONEST_FEATURES)
print(f"HONEST score (5 real features only), n={n_used}: AUC = {honest_auc:.3f}")

# --- THE TRAP: add one column derived straight from the label window ---
panel["LEAK_future_impressions"] = panel["gsc_impressions_lookahead28"]  # literally the label's own input
leaky_auc, n_used_leaky = quick_score(panel, HONEST_FEATURES + ["LEAK_future_impressions"])
print(f"LEAKY score (+ 1 label-derived column), n={n_used_leaky}: AUC = {leaky_auc:.3f}  <-- jumps toward 1.0, exactly the leakage confession from the skill")

# --- delete the leak, keep the honest number ---
panel = panel.drop(columns=["LEAK_future_impressions"])
print(f"\nDeleted the leaky column. Keeping the honest number: AUC = {honest_auc:.3f} (base rate {panel['is_declining'].mean()*100:.1f}%)")


HONEST score (5 real features only), n=48794: AUC = 0.775
LEAKY score (+ 1 label-derived column), n=48794: AUC = 1.000  <-- jumps toward 1.0, exactly the leakage confession from the skill

Deleted the leaky column. Keeping the honest number: AUC = 0.775 (base rate 41.2%)


## 4. Data limits

**Named limitation: the panel is unbalanced across clients, and a global calendar window silently favors the clients with the longest history.** Query (below): of the 104 clients in `dim_clients`, 28 (~27%) have less than 90 days of GSC history as of March 1, 2026 (`DATE_DIFF('day', gsc_data_start, DATE '2026-03-01') < 90`). A `month=2026-03` slice therefore represents whichever clients happened to be tracked that early far more heavily than newer or younger ones -- any pattern I find in this notebook could really be "how FlyRank's longer-tenured clients behave in March," not a lane-wide truth. (Separately: the March 2026 slice itself carries 9,841,378 rows across 331,437 content items but only 55 of the 104 clients have rows in this specific month -- not every client has data in every month, which is its own flavor of the same unbalanced-panel problem.) The fix (per-client window checks against `gsc_data_start`/`ga4_data_start` before trusting any aggregate) is a rule I'm naming now and will actually apply starting at the modeling stage -- not something this contract notebook claims to have already solved.

In [13]:
limitation_check = con.sql("""
    SELECT
        COUNT(*) AS n_clients,
        SUM(CASE WHEN DATE_DIFF('day', gsc_data_start, DATE '2026-03-01') < 90 THEN 1 ELSE 0 END) AS clients_with_under_90d_history_by_march
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
""").df()
limitation_check


,n_clients,clients_with_under_90d_history_by_march
0,104,28.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
